# S4 Builder: LYT-Net TensorFlow (.h5) -> Pre-enhanced Dataset

Notebook ini khusus untuk membangun dataset enhanced S4 dari bobot official .h5.
Output utama: scenarios/S4_LYT_Net/enhanced/

In [ ]:
#@title 0.1 · Environment Setup & Clone Repo
import os, sys, subprocess, shutil

REPO_URL = "https://github.com/Otachiking/Object-Detection-ExDARK-with-LLIE.git"
SCENARIO_KEY = "s4_lyt_net"
SCENARIO_NAME = "S4_LYT_Net"

_IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
_IS_COLAB = (not _IS_KAGGLE) and os.path.exists("/content")

if _IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/TA-IQBAL-ObjectDetectionExDARKwithLLIE"
elif _IS_KAGGLE:
    REPO_DIR = "/kaggle/working/TA-IQBAL-ObjectDetectionExDARKwithLLIE"
else:
    raise RuntimeError("Jalankan notebook ini di Colab atau Kaggle.")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
else:
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Repo ready:", REPO_DIR)

In [ ]:
#@title 0.2 · Install Dependencies (TensorFlow + Utils)
# TensorFlow 2.15 tidak tersedia pada Python 3.12 runtime terbaru.
# Gunakan range yang kompatibel dan stabil untuk notebook ini.
!pip install -q "tensorflow>=2.19,<2.20" opencv-python-headless tqdm pyyaml gdown

import tensorflow as tf
print("TensorFlow:", tf.__version__)

# Guard: pastikan versi yang dipakai sesuai dengan range notebook.
_tf_major, _tf_minor = (int(x) for x in tf.__version__.split(".")[:2])
assert (_tf_major, _tf_minor) == (2, 19), (
    f"Expected TensorFlow 2.19.x, got {tf.__version__}"
)

In [ ]:
#@title 0.3 · Resolve Paths
import os
from src.config import load_config

cfg = load_config(SCENARIO_KEY, quick_test=False)
paths = cfg.get("paths", {})
data_paths = paths.get("data", {})

OUTPUT_ROOT = paths.get("output_root") or paths.get("drive_root") or paths.get("project_root")
EXDARK_ROOT = paths.get("exdark_root") or data_paths.get("exdark_original")

if _IS_KAGGLE:
    from src.utils.platform import resolve_kaggle_exdark
    if EXDARK_ROOT is None or not os.path.isfile(os.path.join(EXDARK_ROOT or "", "Groundtruth", "imageclasslist.txt")):
        EXDARK_ROOT = resolve_kaggle_exdark()

if OUTPUT_ROOT is None: raise KeyError("Cannot resolve OUTPUT_ROOT")
if EXDARK_ROOT is None: raise KeyError("Cannot resolve EXDARK_ROOT")

PREPARED_DIR = os.path.join(OUTPUT_ROOT, "prepared")
YOLO_DIR = os.path.join(PREPARED_DIR, "ExDark_yolo")
SCENARIO_DIR = os.path.join(OUTPUT_ROOT, "scenarios", SCENARIO_NAME)
ENHANCED_DIR = os.path.join(SCENARIO_DIR, "enhanced")

assert os.path.isfile(os.path.join(YOLO_DIR, "dataset.yaml")), "Jalankan Fase 1 notebook S4 dulu."
os.makedirs(ENHANCED_DIR, exist_ok=True)

print("OUTPUT_ROOT :", OUTPUT_ROOT)
print("YOLO_DIR    :", YOLO_DIR)
print("ENHANCED_DIR:", ENHANCED_DIR)

In [ ]:
#@title 1.0 · Load Official LYT-Net TensorFlow + .h5 Weights
import os, sys, subprocess
import tensorflow as tf

LYT_REPO_URL = "https://github.com/albrateanu/LYT-Net.git"
MODEL_CACHE_DIR = os.path.join(OUTPUT_ROOT, "model_cache")
LYT_TF_DIR = os.path.join(MODEL_CACHE_DIR, "LYT-Net-TF")

if not os.path.isdir(os.path.join(LYT_TF_DIR, ".git")):
    os.makedirs(MODEL_CACHE_DIR, exist_ok=True)
    subprocess.run(["git", "clone", LYT_REPO_URL, LYT_TF_DIR], check=True)

tf_root = os.path.join(LYT_TF_DIR, "TensorFlow")
if tf_root not in sys.path:
    sys.path.insert(0, tf_root)
from model.arch import LYT, Denoiser

WEIGHTS_H5_PATH = os.path.join(REPO_DIR, "llie-weights", "lyt_net_LOL_v1.h5")  # @param {type:"string"}
if _IS_KAGGLE and not os.path.isfile(WEIGHTS_H5_PATH):
    _kaggle_h5 = "/kaggle/input/llie-model-cache/lyt_net_LOL_v1.h5"
    if os.path.isfile(_kaggle_h5):
        WEIGHTS_H5_PATH = _kaggle_h5

if not os.path.isfile(WEIGHTS_H5_PATH):
    raise FileNotFoundError(f"Weights .h5 not found: {WEIGHTS_H5_PATH}")

denoiser_cb = Denoiser(16)
denoiser_cr = Denoiser(16)
model = LYT(filters=32, denoiser_cb=denoiser_cb, denoiser_cr=denoiser_cr)

# Official upstream flow first (TensorFlow/scripts/test.py style).
try:
    denoiser_cb.build(input_shape=(None, None, None, 1))
    denoiser_cr.build(input_shape=(None, None, None, 1))
    model.build(input_shape=(None, None, None, 3))
    model.load_weights(WEIGHTS_H5_PATH)
    print("Loaded via official build+load flow")
except Exception as e:
    print(f"Official flow failed: {type(e).__name__}: {e}")
    print("Retry with Keras-compat fallback (materialize variables via forward pass)...")

    # Keras 3 compatibility fallback: ensure variables exist before load_weights.
    _dummy = tf.zeros((1, 256, 256, 3), dtype=tf.float32)
    _ = model(_dummy, training=False)
    print(f"Materialized vars: {len(model.weights)}")
    model.load_weights(WEIGHTS_H5_PATH)
    print("Loaded via fallback forward-pass flow")

print("Final weights source:")
print(WEIGHTS_H5_PATH)

In [ ]:
#@title 1.1 · Build Enhanced Dataset (train/val/test)
import os
import glob
import cv2
import numpy as np
import tensorflow as tf
from tqdm.auto import tqdm
from src.data.build_yolo_dataset import generate_enhanced_dataset_yaml

def _pad_to_mult8(rgb_img: np.ndarray):
    h, w = rgb_img.shape[:2]
    h8 = ((h + 7) // 8) * 8
    w8 = ((w + 7) // 8) * 8
    pad_bottom = h8 - h
    pad_right = w8 - w
    if pad_bottom == 0 and pad_right == 0:
        return rgb_img, h, w

    # Reflect padding menjaga konten tepi lebih natural untuk enhancement.
    padded = cv2.copyMakeBorder(
        rgb_img, 0, pad_bottom, 0, pad_right, borderType=cv2.BORDER_REFLECT
    )
    return padded, h, w

src_images_root = os.path.join(YOLO_DIR, "images")
total = 0

for split in ["train", "val", "test"]:
    in_dir = os.path.join(src_images_root, split)
    out_dir = os.path.join(ENHANCED_DIR, "images", split)
    os.makedirs(out_dir, exist_ok=True)

    img_paths = sorted(glob.glob(os.path.join(in_dir, "*.*")))
    for p in tqdm(img_paths, desc=f"Enhancing {split}", unit="img"):
        fname = os.path.basename(p)
        dst = os.path.join(out_dir, fname)

        if os.path.isfile(dst):
            continue

        bgr = cv2.imread(p, cv2.IMREAD_COLOR)
        if bgr is None:
            continue

        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        rgb_pad, h0, w0 = _pad_to_mult8(rgb)

        x = (rgb_pad.astype(np.float32) / 127.5) - 1.0
        x = np.expand_dims(x, axis=0)
        x = tf.convert_to_tensor(x, dtype=tf.float32)

        y = model(x, training=False).numpy()[0]
        y = np.clip((y + 1.0) * 127.5, 0, 255).astype(np.uint8)

        # Crop balik ke resolusi asli image input.
        y = y[:h0, :w0, :]

        y_bgr = cv2.cvtColor(y, cv2.COLOR_RGB2BGR)
        cv2.imwrite(dst, y_bgr)
        total += 1

dataset_yaml = os.path.join(ENHANCED_DIR, "dataset.yaml")
generate_enhanced_dataset_yaml(
    enhanced_images_dir=ENHANCED_DIR,
    yolo_labels_dir=YOLO_DIR,
    output_yaml_path=dataset_yaml,
)

print("Done. Newly enhanced images:", total)
print("Enhanced dataset ready:", ENHANCED_DIR)
print("dataset.yaml:", dataset_yaml)

In [ ]:
#@title 1.2 · Optional Export to Google Drive (zip)
import os, shutil

ZIP_TO_DRIVE = True  # @param {type:"boolean"}
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/KULIAH-S1INFORMATIKA/TA-IQBAL/S4_H5_Enhanced"  # @param {type:"string"}

if ZIP_TO_DRIVE and _IS_COLAB:
    os.makedirs(DRIVE_EXPORT_DIR, exist_ok=True)
    zip_base = os.path.join("/content", "S4_LYT_Net_enhanced")
    zip_path = shutil.make_archive(zip_base, "zip", root_dir=ENHANCED_DIR)
    dst = os.path.join(DRIVE_EXPORT_DIR, os.path.basename(zip_path))
    shutil.copy2(zip_path, dst)
    print("Exported zip to:", dst)
elif ZIP_TO_DRIVE and not _IS_COLAB:
    print("Skip: export to Drive tersedia di Colab.")
else:
    print("ZIP_TO_DRIVE=False, no export performed.")